# BITS Pilani WILP — Machine Learning Assignment 2
## Mobile Price Classification: Model Training & Fine-Tuning Pipeline

**Author:** Ajay Srivastava  
**Dataset:** [Mobile Price Classification — Kaggle](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification)  
**Dataset Size:** 2,000 instances, 20 features, Target: `price_range` (4 classes)  

This notebook demonstrates the complete end-to-end ML pipeline for the Mobile Price Classification problem.
We train and hyperparameter-tune 5 classification models using 5-Fold Cross Validation:

1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **K-Nearest Neighbors (KNN) Classifier**
4. **Gaussian Naive Bayes**
5. **Random Forest Classifier (Ensemble)**

**6 Compulsory Evaluation Metrics computed for each model:**
- Accuracy
- One-vs-Rest Macro AUC Score
- Precision (Macro)
- Recall (Macro)
- F1-Score (Macro)
- Matthews Correlation Coefficient (MCC)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

# Custom seed derived from author ID: AjaySrivastava13071993
SEED = 1307

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
print(f'Random Seed: {SEED}')

### 1. Load Dataset & Exploratory Analysis

In [ ]:
# Load official Kaggle dataset (Mobile Price Classification)
df = pd.read_csv('../mobile_data.csv')
print(f'Dataset shape: {df.shape}')
print(f'Features: {df.shape[1]-1} | Instances: {df.shape[0]}')
df.head()

In [ ]:
# Class distribution — must be balanced
print('Class distribution in price_range:')
print(df['price_range'].value_counts().sort_index())
print()
df.info()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

### 2. Data Preparation & Feature Scaling

In [ ]:
# Separate features and target variable
X = df.drop('price_range', axis=1)
y = df['price_range']

# Stratified 80/20 train-test split with custom seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f'Training set: {X_train.shape}')
print(f'Test set:     {X_test.shape}')

# StandardScaler — Z-score normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, 'mobile_price_scaler.joblib')
print('Scaler saved.')

### 3. Hyperparameter Tuning & Model Training (5-Fold Cross Validation)

In [ ]:
# Custom hyperparameter grids for each model
model_configurations = {
    'Logistic Regression': {
        'estimator': LogisticRegression(max_iter=1500, random_state=SEED),
        'param_grid': {
            'C': [0.05, 0.5, 2.0, 15.0, 50.0, 100.0],
            'solver': ['lbfgs', 'saga']
        },
        'save_filename': 'logistic_regression_model.joblib'
    },
    'Decision Tree': {
        'estimator': DecisionTreeClassifier(random_state=SEED),
        'param_grid': {
            'max_depth': [4, 6, 9, 14, None],
            'min_samples_split': [2, 4, 8],
            'criterion': ['gini', 'entropy']
        },
        'save_filename': 'decision_tree_model.joblib'
    },
    'kNN': {
        'estimator': KNeighborsClassifier(),
        'param_grid': {
            'n_neighbors': [5, 9, 13, 17, 21],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan', 'minkowski']
        },
        'save_filename': 'knn_classifier_model.joblib'
    },
    'Naive Bayes': {
        'estimator': GaussianNB(),
        'param_grid': {
            'var_smoothing': np.logspace(-1, -9, num=50)
        },
        'save_filename': 'naive_bayes_model.joblib'
    },
    'Random Forest': {
        'estimator': RandomForestClassifier(random_state=SEED),
        'param_grid': {
            'n_estimators': [80, 120, 200],
            'max_depth': [6, 12, None],
            'min_samples_split': [2, 6],
            'criterion': ['gini', 'entropy']
        },
        'save_filename': 'random_forest_model.joblib'
    }
}

benchmark_results = []
confusion_matrices = {}

print(f'--- 5-Fold GridSearchCV Tuning (seed={SEED}) ---')
for model_name, config in model_configurations.items():
    print(f'\nTuning {model_name}...')
    grid_search = GridSearchCV(
        estimator=config['estimator'],
        param_grid=config['param_grid'],
        cv=5, scoring='accuracy', n_jobs=-1
    )
    grid_search.fit(X_train_scaled, y_train)
    best_model = grid_search.best_estimator_
    print(f'  Best params: {grid_search.best_params_}')

    # Save model artifact
    joblib.dump(best_model, config['save_filename'])

    # Predict on holdout test set
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)

    # Calculate all 6 required metrics
    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)
    mcc  = matthews_corrcoef(y_test, y_pred)

    benchmark_results.append({
        'ML Model Name': model_name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })
    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred)
    print(f'  Acc={acc:.4f} | AUC={auc:.4f} | F1={f1:.4f} | MCC={mcc:.4f}')

### 4. Evaluation Metrics — Comparison Table

In [ ]:
df_results = pd.DataFrame(benchmark_results)
df_results.to_csv('model_metrics.csv', index=False)
print('Metrics saved to model_metrics.csv')
df_results

### 5. Model Performance Comparison Chart

In [ ]:
df_melted = df_results.melt(id_vars='ML Model Name', var_name='Metric', value_name='Score')
plt.figure(figsize=(13, 6))
sns.barplot(data=df_melted, x='ML Model Name', y='Score', hue='Metric')
plt.title('Benchmark Performance Across 6 Evaluation Metrics', fontsize=14, fontweight='bold')
plt.ylim(0.4, 1.05)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 6. Confusion Matrices for All Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.ravel()

for i, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
    axes[i].set_title(f'Confusion Matrix — {name}', fontweight='bold')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')

fig.delaxes(axes[5])
plt.suptitle('Confusion Matrices — All 5 Models', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 7. Observations & Winner

| ML Model Name | Observation |
|:---|:---|
| **Logistic Regression** | Best performer. Strong linear separability between `ram`/pixel features and price tier gives near-perfect accuracy (98%) and AUC (0.9995). |
| **Random Forest** | Second best (88.75%). Ensemble of trees captures non-linear interactions effectively with low variance. |
| **Decision Tree** | 83% accuracy. Interpretable but higher variance than ensemble methods; some boundary misclassification. |
| **Naive Bayes** | 79% accuracy. Feature independence assumption limits performance but AUC (0.9495) remains high. |
| **kNN** | Weakest at 67.5%. Curse of dimensionality and mixed binary/continuous features reduce distance quality. |
| **Overall Winner** | **Logistic Regression** — 98.00% accuracy, 0.9995 AUC, 0.9734 MCC. |